# Evaluación 2 — ¿Ayuda arrancar la red en la base de Fourier?

**Programación Científica 2026-1 · Universidad Nacional de Colombia (Medellín)**

**Estudiante:** Santiago Abelardo Salcedo rodriguez · **ID (últimos 4 dígitos):** `3213`

---

## Pregunta que guía el trabajo

Aproximamos una señal como combinación de bases, $\hat f(t)=\sum_i c_i\phi_i(t)$. Una red
neuronal hace lo mismo pero **aprende** sus bases. Construimos un autoencoder que
**reconstruye** una señal,

$$\hat x = W_2\,\tanh(W_1 x),$$

y estudiamos qué pasa cuando la primera capa $W_1$ **arranca como la Transformada de
Fourier** en lugar de una base aleatoria. La pregunta central:
*¿conviene empezar en la base "correcta" (Fourier) o da igual arrancar aleatorio?*

### Parámetros personalizados (ID 3213)

| Parámetro | Valor |
|---|---|
| $K$ (frecuencias) | **25** → capa $256\to 50$ |
| Iteraciones | **2000** |
| Learning rate $\eta$ | **0.05** |
| Señales por régimen | **8** (24 en total) |
| Régimen foco | **1** ($\mu=5$, dispersión intermedia) |
| Umbral de compresión | **95 %** de la energía |

### La física de los datos

Cada señal es una suma de 3 cosenos, $f(t)=\sum_j a_j\cos(\omega(k_j)\,2\pi t-k_j s+\varphi_j)$,
con la **regla de dispersión** $\omega(k)=\sqrt{c^2k^2+\mu^2}$, $c=1$. El parámetro $\mu$
define tres regímenes:

- **Régimen 0** ($\mu=0$): $\omega=k$ → **frecuencias enteras**, que caen *exactamente* sobre
  la rejilla de Fourier $\{\sin(2\pi kt),\cos(2\pi kt)\}$.
- **Régimen 1** ($\mu=5$) y **Régimen 2** ($\mu=15$): $\omega=\sqrt{k^2+\mu^2}$ es **no entera**
  → las frecuencias caen *entre* los puntos de la rejilla (fuga espectral) y suben con $\mu$.

El enunciado pide prestar especial atención al **régimen 1**, donde la ventaja de Fourier
debería ser intermedia: hay dispersión pero las frecuencias no son tan altas como en el régimen 2.

## 0 · Preparación: librerías, estilo de figuras y carga de datos

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# JAX en float64 para comparar con precisión la capa Fourier vs np.fft
jax.config.update("jax_enable_x64", True)

SEMILLA = 3213

# --- Paleta de visualización (Okabe–Ito, segura para daltonismo) ---
# Los tres regímenes son categorías → colores fijos en orden fijo.
C_REG = {
    0: "#0072B2",   # azul — régimen 0 (sin dispersión)
    1: "#D55E00",   # bermellón — régimen 1 (foco del enunciado)
    2: "#009E73",   # verde — régimen 2 (dispersión fuerte)
}
NOMBRE_REG = {
    0: r"Régimen 0 (μ=0, sin dispersión)",
    1: r"Régimen 1 (μ=5, intermedia)",
    2: r"Régimen 2 (μ=15, fuerte)",
}
C_FOURIER, C_RANDOM = "#0072B2", "#D55E00"
LS_FOURIER, LS_RANDOM = "-", "--"

# --- Estilo global de figuras ---
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 13, "axes.titlesize": 14, "axes.titleweight": "bold",
    "legend.frameon": False, "figure.autolayout": True,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "STIXGeneral", "Times New Roman"],
})

print("JAX", jax.__version__, "· float64:", jnp.zeros(1).dtype)

ModuleNotFoundError: No module named 'jax'

### Carga de datos e inspección

In [ ]:
datos = np.load("datos_3213.npz")
X = np.asarray(datos["X"])
y = np.asarray(datos["y"]).astype(int)
t = np.asarray(datos["t"])
meta = list(datos["meta"])

N = X.shape[1]                       # 256 muestras temporales
K = 25                                # frecuencias de la capa Fourier
ITERS = 2000                          # iteraciones de entrenamiento
LR = 0.05                             # learning rate
REGIMENES = [0, 1, 2]

print(f"X: {X.shape}   y: {y.shape}   t: {t.shape}")
print(f"meta [ID, huella, timestamp]: {meta}")
print(f"muestras por régimen: {[(y==r).sum() for r in REGIMENES]}")
print(f"t ∈ [{t.min():.3f}, {t.max():.3f}]  (Δt = {t[1]-t[0]:.5f}, dominio normalizado)")

# --- Energía media del dataset (entregable de la rúbrica, 3 decimales) ---
energia_media = float((X**2).mean())
print(f"\n>>> Energía media del dataset  (X**2).mean() = {energia_media:.3f}")
for r in REGIMENES:
    print(f"    energía media régimen {r}: {(X[y==r]**2).mean():.3f}")

### Un vistazo a los datos

Antes de modelar, miramos una señal de cada régimen. La forma temporal ya delata la
física: el régimen 0 oscila lento (frecuencias enteras bajas), y la frecuencia
aparente **crece con $\mu$** (regímenes 1 y 2).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

for r, ax in zip(REGIMENES, axes):
    i = np.where(y == r)[0][0]
    ax.axhline(0, color="0.7", lw=0.6)
    ax.plot(t, X[i], color=C_REG[r], lw=1.4)
    ax.set_ylabel("amplitud")

    lo, hi = X[i].min(), X[i].max()
    rango = hi - lo
    ax.set_ylim(lo - 0.10*rango, hi + 0.38*rango)

    ax.text(0.012, 0.955, NOMBRE_REG[r], transform=ax.transAxes,
            color=C_REG[r], fontweight="bold", va="top")

axes[-1].set_xlabel("t  (tiempo, unidad normalizada)")
fig.suptitle("Una señal representativa por régimen", fontweight="bold", y=0.98)
plt.show()